# Fase 1: Setup

In [306]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import skew, chi2_contingency

In [307]:
df = pd.read_csv("db.csv")

In [308]:
# Identificación de tipos de datos.
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27901 entries, 0 to 27900
Data columns (total 18 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   id                                     27901 non-null  int64  
 1   Gender                                 27901 non-null  object 
 2   Age                                    27901 non-null  float64
 3   City                                   27901 non-null  object 
 4   Profession                             27901 non-null  object 
 5   Academic Pressure                      27901 non-null  float64
 6   Work Pressure                          27901 non-null  float64
 7   CGPA                                   27901 non-null  float64
 8   Study Satisfaction                     27901 non-null  float64
 9   Job Satisfaction                       27901 non-null  float64
 10  Sleep Duration                         27901 non-null  object 
 11  Di

In [309]:
print(f"Cantidad de objetos en la base de datos: {df.shape[0]} objetos.")
print(f"Cantidad de columnas en la base de datos: {df.shape[1]} columnas.")

Cantidad de objetos en la base de datos: 27901 objetos.
Cantidad de columnas en la base de datos: 18 columnas.


## Análisis exploratorio de datos

In [310]:
df.describe()

,id,Age,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Work/Study Hours,Financial Stress,Depression
count,27901.000000,27901.000000,27901.000000,27901.000000,27901.000000,27901.000000,27901.000000,27901.000000,27898.000000,27901.000000
mean,70442.149421,25.822300,3.141214,0.000430,7.656104,2.943837,0.000681,7.156984,3.139867,0.585499
std,40641.175216,4.905687,1.381465,0.043992,1.470707,1.361148,0.044394,3.707642,1.437347,0.492645
min,2.000000,18.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
25%,35039.000000,21.000000,2.000000,0.000000,6.290000,2.000000,0.000000,4.000000,2.000000,0.000000
50%,70684.000000,25.000000,3.000000,0.000000,7.770000,3.000000,0.000000,8.000000,3.000000,1.000000
75%,105818.000000,30.000000,4.000000,0.000000,8.920000,4.000000,0.000000,10.000000,4.000000,1.000000
max,140699.000000,59.000000,5.000000,5.000000,10.000000,5.000000,4.000000,12.000000,5.000000,1.000000


In [311]:
# Funciones de detección de outliers
def detectar_outliers_iqr(df):

    resultados = {}

    columnas_numericas = df.select_dtypes(include='number').columns

    for col in columnas_numericas:

        serie = df[col].dropna()

        Q1 = serie.quantile(0.25)
        Q3 = serie.quantile(0.75)
        IQR = Q3 - Q1

        limite_inferior = Q1 - 1.5 * IQR
        limite_superior = Q3 + 1.5 * IQR

        mascara = (serie < limite_inferior) | (serie > limite_superior)

        resultados[col] = {
            "skew": serie.skew(),
            "cantidad_outliers": mascara.sum(),
            "limite_inferior": limite_inferior,
            "limite_superior": limite_superior
        }

    return resultados

In [312]:
def detectar_outliers_3sigmas(df):

    resultados = {}

    columnas_numericas = df.select_dtypes(include='number').columns

    for col in columnas_numericas:

        serie = df[col].dropna()

        media = serie.mean()
        std = serie.std()

        limite_inferior = media - 3 * std
        limite_superior = media + 3 * std

        mascara = (serie < limite_inferior) | (serie > limite_superior)

        resultados[col] = {
            "skew": serie.skew(),
            "cantidad_outliers": mascara.sum(),
            "limite_inferior": limite_inferior,
            "limite_superior": limite_superior
        }

    return resultados

In [313]:
def eliminar_outliers_3sigmas(serie):
    media = serie.mean()
    std = serie.std()

    limite_inferior = media - 3 * std
    limite_superior = media + 3 * std

    mascara = (serie < limite_inferior) | (serie > limite_superior)

    return mascara, limite_inferior, limite_superior

In [314]:
def detectar_duplicados(df):

    resultados = {}

    for col in df.columns:

        duplicados = df[col].duplicated().sum()

        resultados[col] = {
            "cantidad_duplicados": duplicados
        }

    return resultados

In [315]:
outliers_iqr = detectar_outliers_iqr(df)

In [316]:
outliers_3sig = detectar_outliers_3sigmas(df)

In [317]:
duplicados = detectar_duplicados(df)

In [318]:
for col, info in outliers_3sig.items():

    print("Detección de outliers con 3 sigmas")
    print(f"Columna: {col}")
    print(f"Asimetría (skew): {info['skew']:.3f}")
    print(f"Cantidad de outliers: {info['cantidad_outliers']}")
    print(f"Límite inferior: {info['limite_inferior']:.3f}")
    print(f"Límite superior: {info['limite_superior']:.3f}")
    print("-"*40)

Detección de outliers con 3 sigmas
Columna: id
Asimetría (skew): -0.005
Cantidad de outliers: 0
Límite inferior: -51481.376
Límite superior: 192365.675
----------------------------------------
Detección de outliers con 3 sigmas
Columna: Age
Asimetría (skew): 0.132
Cantidad de outliers: 19
Límite inferior: 11.105
Límite superior: 40.539
----------------------------------------
Detección de outliers con 3 sigmas
Columna: Academic Pressure
Asimetría (skew): -0.135
Cantidad de outliers: 0
Límite inferior: -1.003
Límite superior: 7.286
----------------------------------------
Detección de outliers con 3 sigmas
Columna: Work Pressure
Asimetría (skew): 108.594
Cantidad de outliers: 3
Límite inferior: -0.132
Límite superior: 0.132
----------------------------------------
Detección de outliers con 3 sigmas
Columna: CGPA
Asimetría (skew): -0.113
Cantidad de outliers: 9
Límite inferior: 3.244
Límite superior: 12.068
----------------------------------------
Detección de outliers con 3 sigmas
Colum

In [319]:
for col, info in outliers_iqr.items():

    print("Outliers detectados con IQR")
    print(f"Columna: {col}")
    print(f"Asimetría (skew): {info['skew']:.3f}")
    print(f"Cantidad de outliers: {info['cantidad_outliers']}")
    print(f"Límite inferior: {info['limite_inferior']:.3f}")
    print(f"Límite superior: {info['limite_superior']:.3f}")
    print("-"*40)

Outliers detectados con IQR
Columna: id
Asimetría (skew): -0.005
Cantidad de outliers: 0
Límite inferior: -71129.500
Límite superior: 211986.500
----------------------------------------
Outliers detectados con IQR
Columna: Age
Asimetría (skew): 0.132
Cantidad de outliers: 12
Límite inferior: 7.500
Límite superior: 43.500
----------------------------------------
Outliers detectados con IQR
Columna: Academic Pressure
Asimetría (skew): -0.135
Cantidad de outliers: 0
Límite inferior: -1.000
Límite superior: 7.000
----------------------------------------
Outliers detectados con IQR
Columna: Work Pressure
Asimetría (skew): 108.594
Cantidad de outliers: 3
Límite inferior: 0.000
Límite superior: 0.000
----------------------------------------
Outliers detectados con IQR
Columna: CGPA
Asimetría (skew): -0.113
Cantidad de outliers: 9
Límite inferior: 2.345
Límite superior: 12.865
----------------------------------------
Outliers detectados con IQR
Columna: Study Satisfaction
Asimetría (skew): 0.0

In [320]:
print("Detección de duplicados por columna\n")

for col, info in duplicados.items():

    print(f"Columna: {col}")
    print(f"Cantidad de duplicados: {info['cantidad_duplicados']}")
    print("-"*40)

Detección de duplicados por columna

Columna: id
Cantidad de duplicados: 0
----------------------------------------
Columna: Gender
Cantidad de duplicados: 27899
----------------------------------------
Columna: Age
Cantidad de duplicados: 27867
----------------------------------------
Columna: City
Cantidad de duplicados: 27849
----------------------------------------
Columna: Profession
Cantidad de duplicados: 27887
----------------------------------------
Columna: Academic Pressure
Cantidad de duplicados: 27895
----------------------------------------
Columna: Work Pressure
Cantidad de duplicados: 27898
----------------------------------------
Columna: CGPA
Cantidad de duplicados: 27569
----------------------------------------
Columna: Study Satisfaction
Cantidad de duplicados: 27895
----------------------------------------
Columna: Job Satisfaction
Cantidad de duplicados: 27896
----------------------------------------
Columna: Sleep Duration
Cantidad de duplicados: 27896
----------

In [321]:
duplicados = df[df.duplicated() == True]

duplicados

,id,Gender,Age,City,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,Depression


In [322]:
df['City'].unique()

array(['Visakhapatnam', 'Bangalore', 'Srinagar', 'Varanasi', 'Jaipur',
       'Pune', 'Thane', 'Chennai', 'Nagpur', 'Nashik', 'Vadodara',
       'Kalyan', 'Rajkot', 'Ahmedabad', 'Kolkata', 'Mumbai', 'Lucknow',
       'Indore', 'Surat', 'Ludhiana', 'Bhopal', 'Meerut', 'Agra',
       'Ghaziabad', 'Hyderabad', 'Vasai-Virar', 'Kanpur', 'Patna',
       'Faridabad', 'Delhi', 'Saanvi', 'M.Tech', 'Bhavna', 'Less Delhi',
       'City', '3.0', 'Less than 5 Kalyan', 'Mira', 'Harsha', 'Vaanya',
       'Gaurav', 'Harsh', 'Reyansh', 'Kibara', 'Rashi', 'ME', 'M.Com',
       'Nalyan', 'Mihir', 'Nalini', 'Nandini', 'Khaziabad'], dtype=object)

In [323]:
df['Job Satisfaction'].value_counts()

,count
Job Satisfaction,
0.0,27893
2.0,3
4.0,2
1.0,2
3.0,1


In [324]:
df['Financial Stress'].value_counts()

,count
Financial Stress,
5.0,6715
4.0,5775
3.0,5226
1.0,5121
2.0,5061


# Etapa de limpieza

In [325]:
df_copia = df.copy()

In [326]:
columnas_numericas = df_copia.select_dtypes(include='number').columns

for col in columnas_numericas:

    mascara, li, ls = eliminar_outliers_3sigmas(df_copia[col])

    print(f"Procesando columna: {col}")
    print(f"Outliers detectados: {mascara.sum()}")

    df_copia = df_copia[~mascara]

    print("-"*40)

Procesando columna: id
Outliers detectados: 0
----------------------------------------
Procesando columna: Age
Outliers detectados: 19
----------------------------------------
Procesando columna: Academic Pressure
Outliers detectados: 0
----------------------------------------
Procesando columna: Work Pressure
Outliers detectados: 3
----------------------------------------
Procesando columna: CGPA
Outliers detectados: 6
----------------------------------------
Procesando columna: Study Satisfaction
Outliers detectados: 0
----------------------------------------
Procesando columna: Job Satisfaction
Outliers detectados: 2
----------------------------------------
Procesando columna: Work/Study Hours
Outliers detectados: 0
----------------------------------------
Procesando columna: Financial Stress
Outliers detectados: 0
----------------------------------------
Procesando columna: Depression
Outliers detectados: 0
----------------------------------------


In [327]:
print("Filas originales:", len(df))
print("Filas después de eliminar outliers:", len(df_copia))

Filas originales: 27901
Filas después de eliminar outliers: 27871


In [328]:
len(df_copia) - len(df)

-30

In [329]:
df_copia = df_copia.dropna()

In [330]:
print("Filas originales:", len(df))
print("Filas después de dropear nulos:", len(df_copia))

Filas originales: 27901
Filas después de dropear nulos: 27868


In [331]:
len(df_copia) - len(df)

-33